# 03a_batch_report: Reproducible batch pipeline (CLI)

Goal: run a small, reproducible Polars batch job that reads the generated wearable tables, joins them, and writes a report artifact.

In notebook form, we’ll implement the same pipeline in cells (using the same config), then display the output table and an inline plot.

This is the same mental model as a real lab pipeline: config-driven inputs, lazy scans, and streaming output.

This demo has two representations:

- The implementation is the script: `03a_batch_report.py`.
- This notebook is a walkthrough with additional commentary.

## Files

- `03a_batch_report.py` (script)
- `03a_config.yaml` (config)

## Data setup

We’ll generate the dataset if it’s missing.

In [1]:
from pathlib import Path
import subprocess
import sys

Path("data").mkdir(parents=True, exist_ok=True)
Path("outputs").mkdir(parents=True, exist_ok=True)

sensor_dir = Path("data/sensor_hrv")
if not sensor_dir.exists() or not list(sensor_dir.glob("*.parquet")):
    subprocess.run(
        [sys.executable, "generate_demo_data.py", "--size", "small", "--output-dir", "data"],
        check=True,
    )

## 1) Configure inputs/filters/outputs (YAML)

We’ll read `03a_config.yaml` so the paths and filters are not hardcoded.

In [2]:
from pathlib import Path
import yaml

cfg = yaml.safe_load(Path("03a_config.yaml").read_text())
cfg

{'inputs': {'sensor_parquet': 'data/sensor_hrv/*.parquet',
  'sleep_parquet': 'data/sleep_diary.parquet',
  'users_parquet': 'data/user_profile.parquet'},
 'filters': {'start_date': '2024-01-15',
  'missingness_max': 0.35,
  'night_hours': {'start': 22, 'end': 6}},
 'outputs': {'report_parquet': 'outputs/sleep_hrv_report.parquet',
  'report_csv': 'outputs/sleep_hrv_report.csv'}}

## 2) Load phase (lazy scans)

The important point is that all three tables are scanned lazily.

In [3]:
import polars as pl

sensor = pl.scan_parquet(cfg["inputs"]["sensor_parquet"])
sleep = pl.scan_parquet(cfg["inputs"]["sleep_parquet"])
users = pl.scan_parquet(cfg["inputs"]["users_parquet"])

sensor.collect_schema()

Schema([('device_id', String),
        ('ts_start', Datetime(time_unit='us', time_zone=None)),
        ('ts_end', Datetime(time_unit='us', time_zone=None)),
        ('heart_rate', Int64),
        ('hrv_sdnn', Float64),
        ('hrv_rmssd', Float64),
        ('hrv_lf_hf_ratio', Float64),
        ('bvp_mean', Float64),
        ('spo2', Float64),
        ('eda_mean', Float64),
        ('skin_temp', Float64),
        ('acc_magnitude', Float64),
        ('steps', Int64),
        ('missingness_score', Float64)])

## 3) Transform phase (keys → aggregate → joins)

We’ll:

1. filter the big sensor table early
2. derive `user_id` + `date` so it can join to `sleep_diary`
3. aggregate to one row per user-night (this keeps the join small)

In [4]:
from datetime import datetime

start_dt = datetime.fromisoformat(cfg["filters"]["start_date"])
missingness_max = float(cfg["filters"]["missingness_max"])

sensor_keyed = (
    sensor
    .filter(pl.col("missingness_score") <= missingness_max)
    .filter(pl.col("ts_start") >= start_dt)
    .with_columns(
        pl.concat_str([pl.lit("USER-"), pl.col("device_id").str.split("-").list.get(1)]).alias("user_id"),
        pl.col("ts_start").dt.date().alias("date"),
        pl.col("ts_start").dt.hour().alias("hour"),
    )
)

sensor_night = sensor_keyed.filter((pl.col("hour") >= 22) | (pl.col("hour") <= 6)).select(
    ["user_id", "date", "heart_rate", "hrv_sdnn", "hrv_rmssd", "steps"]
)

nightly = sensor_night.group_by(["user_id", "date"]).agg(
    [
        pl.len().alias("num_segments"),
        pl.mean("heart_rate").alias("night_mean_hr"),
        pl.mean("hrv_sdnn").alias("night_mean_sdnn"),
        pl.mean("hrv_rmssd").alias("night_mean_rmssd"),
        pl.sum("steps").alias("night_steps"),
    ]
)

joined = (
    sleep.select(["user_id", "date", "sleep_efficiency"])
    .join(nightly, on=["user_id", "date"], how="inner")
    .join(users.select(["user_id", "age", "gender", "occupation"]), on="user_id", how="left")
)

report = (
    joined.group_by(["occupation", "gender"]).agg(
        [
            pl.len().alias("n_nights"),
            pl.mean("sleep_efficiency").alias("avg_sleep_efficiency"),
            pl.mean("night_mean_sdnn").alias("avg_night_sdnn"),
            pl.corr("sleep_efficiency", "night_mean_sdnn").alias("corr_sleep_sdnn"),
        ]
    )
    .sort(["occupation", "gender"])
)

print(report.explain())

SORT BY [col("occupation"), col("gender")]
  AGGREGATE[maintain_order: false]
    [len().alias("n_nights"), col("sleep_efficiency").mean().alias("avg_sleep_efficiency"), col("night_mean_sdnn").mean().alias("avg_night_sdnn"), col("sleep_efficiency").pearson_correlation([col("night_mean_sdnn")]).alias("corr_sleep_sdnn")] BY [col("occupation"), col("gender")]
    FROM
    simple π 4/4 ["sleep_efficiency", ... 3 other columns]
      LEFT JOIN:
      LEFT PLAN ON: [col("user_id")]
        simple π 3/3 ["user_id", "sleep_efficiency", ... 1 other column]
          INNER JOIN:
          LEFT PLAN ON: [col("user_id"), col("date")]
            Parquet SCAN [data/sleep_diary.parquet]
            PROJECT 3/9 COLUMNS
            ESTIMATED ROWS: 5250
          RIGHT PLAN ON: [col("user_id"), col("date")]
            AGGREGATE[maintain_order: false]
              [len().alias("num_segments"), col("heart_rate").mean().alias("night_mean_hr"), col("hrv_sdnn").mean().alias("night_mean_sdnn"), col("hrv_rm

## 4) Materialize phase (streaming collect + write artifacts)

In [5]:
out = report.collect(engine="streaming")

parquet_path = Path(cfg["outputs"]["report_parquet"])
csv_path = Path(cfg["outputs"]["report_csv"])

parquet_path.parent.mkdir(parents=True, exist_ok=True)
out.write_parquet(parquet_path)
out.write_csv(csv_path)

out.head(10)

occupation,gender,n_nights,avg_sleep_efficiency,avg_night_sdnn,corr_sleep_sdnn
str,str,u32,f64,f64,f64
"""Graduate""","""Female""",273,97.794139,88.655643,0.028214
"""Graduate""","""Male""",210,97.537619,84.601043,0.022927
"""Graduate""","""Nonbinary""",252,97.649603,90.008951,0.207944
"""Office Worker""","""Female""",273,97.72381,87.360674,-0.027667
"""Office Worker""","""Male""",273,97.651282,87.592452,0.130063
"""Office Worker""","""Nonbinary""",210,98.107143,90.514046,0.109159
"""Other""","""Female""",210,97.904286,89.245786,0.13555
"""Other""","""Male""",399,97.940602,88.10383,0.001834
"""Other""","""Nonbinary""",294,98.014286,86.315457,-0.041619


## 5) Quick validation

In [6]:
assert out.height > 0
assert {"occupation", "gender", "n_nights"}.issubset(set(out.columns))
assert out["n_nights"].min() > 0
assert out["avg_sleep_efficiency"].is_finite().all()
assert out["avg_night_sdnn"].is_finite().all()
"OK: output looks sane"

'OK: output looks sane'

Watch the logs:

- “Scanning inputs lazily…” (load phase)
- “Plan summary…” (transform phase)
- “Collecting with streaming engine…” (materialize phase)
- “Writing …parquet / …csv” (artifact emission)

## 3) Confirm artifacts exist

In [7]:
%%bash
ls -lh outputs/

total 152
-rw-r--r--@ 1 christopher  staff    52K Jan 14 00:04 daily_device_summary.parquet
-rw-r--r

--@ 1 christopher  staff   1.0K Jan 14 00:04 sleep_hrv_by_group.csv
-rw-r--r--@ 1 christopher  staff

   2.5K Jan 14 00:04 sleep_hrv_by_group.parquet
-rw-r--r--@ 1 christopher  staff   2.6K Jan 14 00:04

 sleep_hrv_by_group_sink.parquet
-rw-r--r--@ 1 christopher  staff   1.0K Jan 14 00:04 sleep_hrv_repo

rt.csv
-rw-r--r--@ 1 christopher  staff   2.6K Jan 14 00:04 sleep_hrv_report.parquet


Expected:

- `outputs/sleep_hrv_report.parquet`
- `outputs/sleep_hrv_report.csv`

## 4) Validate the output (fast sanity checks)

Run this immediately after the pipeline finishes:

In [8]:
%%bash
uv run python - <<'PY'
import polars as pl

df = pl.read_parquet("outputs/sleep_hrv_report.parquet")
print(df.head())

assert df.height > 0
assert {"occupation", "gender", "n_nights"}.issubset(set(df.columns))
assert df["n_nights"].min() > 0

# Basic plausibility checks (not strict scientific claims)
assert df["avg_sleep_efficiency"].is_finite().all()
assert df["avg_night_sdnn"].is_finite().all()

print("OK: output looks sane")
PY

shape: (5, 6)
┌───────────────┬───────────

┬──────────┬──────────────────────

┬────────────────┬───────────────

──┐
│ occupation    ┆ gender    ┆ n_nights ┆ avg_sleep_efficiency ┆ avg_night_sdnn 

┆ corr_sleep_sdnn │
│ ---           ┆ ---       ┆ ---      ┆ ---                  ┆ --- 

...

ffice Worker ┆ Male      ┆ 273      ┆ 97.651282            ┆ 87.592452      ┆ 0.130063    

    │
└───────────────┴───────────┴─

─────────┴──────────────────────┴─

───────────────┴─────────────────

┘
OK: output looks sane


## Visual: group summary

In [9]:
import polars as pl
import altair as alt

report = pl.read_parquet("outputs/sleep_hrv_report.parquet")
report.head()

occupation,gender,n_nights,avg_sleep_efficiency,avg_night_sdnn,corr_sleep_sdnn
str,str,u32,f64,f64,f64
"""Graduate""","""Female""",273,97.794139,88.655643,0.028214
"""Graduate""","""Male""",210,97.537619,84.601043,0.022927
"""Graduate""","""Nonbinary""",252,97.649603,90.008951,0.207944
"""Office Worker""","""Female""",273,97.72381,87.360674,-0.027667
"""Office Worker""","""Male""",273,97.651282,87.592452,0.130063


In [10]:
chart_df = report.to_pandas()

alt.Chart(chart_df).mark_bar().encode(
    x=alt.X("occupation:N", sort=None),
    y=alt.Y("avg_night_sdnn:Q", title="Avg night SDNN"),
    color="gender:N",
    tooltip=["n_nights", "avg_sleep_efficiency", "avg_night_sdnn"],
).properties(width=700, height=300)

alt.Chart(...)

## Checkpoints

- The script logs the query plan and output paths.
- The parquet output has one row per `(occupation, gender)` group.
- The report includes both sleep and physiology columns.